# Treinamento — detecção de insetos-praga da soja

TCC de Engenharia da Computação, Centro Universitário IBMEC – RJ
Lucas Muniz da Costa Marques · Orientador: Prof. PhD. William de Souza Barbosa

Executa o procedimento experimental da Seção 2.5.3 do trabalho escrito: a fase 1
compara YOLOv8 n, s e m nas cinco dobras com semente fixa, e a fase 2 repete o
modelo escolhido com duas sementes adicionais.

## Antes de executar

1. Envie `dados_yolo.zip` para o Google Drive, em `MyDrive/TCC/`.
2. **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4.**
   A T4 não é opcional: o capítulo 2 declara que o tempo de inferência em GPU é
   medido nela, e é a essa condição que a meta de 50 ms se refere. Uma L4 ou A100
   invalidaria a comparação.
3. Execute as células em ordem. A partir daí o notebook é idempotente: pode ser
   reexecutado à vontade, e cada execução retoma de onde a sessão anterior parou.

## Por que o notebook retoma

O orçamento estimado é de cerca de 25 horas de GPU, e nenhuma sessão do Colab
dura isso. A sessão cai por inatividade, por limite de uso ou porque o navegador
fechou.

Ao fim de **cada época**, os pesos, o estado do otimizador e as métricas são
copiados para o Drive. Reconectar e executar tudo de novo custa minutos, não um
treino inteiro.

## 1. Ambiente

In [ ]:
import subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())

In [ ]:
# A GPU fica registrada: o capitulo 2 declara a T4, e qualquer outra placa torna
# as medicoes de tempo incomparaveis com o que esta escrito.
GPU = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()

if "T4" not in GPU:
    print(f"ATENCAO: a GPU conectada e '{GPU}', nao uma T4.")
    print("Troque em Ambiente de execucao > Alterar o tipo de ambiente de execucao,")
    print("ou registre a troca no capitulo 2 antes de relatar tempos de inferencia.")
else:
    print(f"OK: {GPU}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# A versao instalada e gravada junto dos resultados: sem ela, o experimento nao e
# reproduzivel, porque a biblioteca calcula a perda, as metricas e a NMS.
VERSAO_ULTRALYTICS = "8.4.183"   # vazio = ultima versao disponivel

alvo = f"ultralytics=={VERSAO_ULTRALYTICS}" if VERSAO_ULTRALYTICS else "ultralytics"
subprocess.run(["pip", "install", "-q", alvo], check=True)

import ultralytics
ultralytics.checks()

## 2. Configuração

In [ ]:
from pathlib import Path

DRIVE     = Path("/content/drive/MyDrive/TCC")
PACOTE    = DRIVE / "dados_yolo.zip"
DADOS     = Path("/content/dados_yolo")      # disco local da VM
RUNS      = Path("/content/runs")            # saida local do treino
RUNS_DRIVE = DRIVE / "runs"                  # espelho persistente

DOBRAS   = [1, 2, 3, 4, 5]
MODELOS  = ["yolov8n", "yolov8s", "yolov8m"]
SEMENTE_FASE1 = 0
SEMENTES_FASE2 = [1, 2]

# Hiperparametros da Tabela da Secao 2.5.2 — identicos para os tres modelos, para
# que a comparacao isole o efeito da capacidade do modelo.
HP = dict(
    imgsz=640, epochs=150, patience=30, batch=16,
    optimizer="SGD", momentum=0.937, lr0=0.01, lrf=0.01,
    warmup_epochs=3, weight_decay=0.0005, cos_lr=False,
    amp=True, deterministic=True, workers=2, val=True, plots=True,
    # aumento de dados da Secao 2.4.4
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.5, degrees=10.0, translate=0.1, scale=0.5,
    mosaic=1.0, close_mosaic=10, mixup=0.0, copy_paste=0.0,
    # 'ram' guarda as imagens decodificadas (~2,6 GB) e encurta a epoca;
    # troque para False se a VM ficar sem memoria.
    cache="ram",
)

RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
print(f"pacote .......... {PACOTE}  (existe: {PACOTE.exists()})")
print(f"espelho ......... {RUNS_DRIVE}")

## 3. Dados

O pacote é copiado do Drive para o disco local da VM e descompactado ali. Ler as
imagens direto do Drive durante o treino é lento o bastante para dominar o tempo
de época.

In [ ]:
import shutil, zipfile, time

if DADOS.exists():
    print(f"{DADOS} ja esta no disco local; nada a fazer.")
else:
    assert PACOTE.exists(), f"envie dados_yolo.zip para {PACOTE.parent}"
    t0 = time.time()
    local_zip = Path("/content/dados_yolo.zip")
    shutil.copy2(PACOTE, local_zip)
    print(f"copiado do Drive em {time.time()-t0:.0f}s")

    t0 = time.time()
    DADOS.mkdir(parents=True)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(DADOS)
    local_zip.unlink()
    print(f"descompactado em {time.time()-t0:.0f}s")

In [ ]:
# Os arquivos de configuracao foram gerados na maquina local e apontam para o
# caminho de la. Aqui so o campo 'path' muda; a particao em si nao e recalculada,
# para que as dobras usadas no treino sejam exatamente as auditadas.
for dobra in DOBRAS:
    arq = DADOS / f"data_dobra{dobra}.yaml"
    texto = arq.read_text(encoding="utf-8")
    novo = "\n".join(
        f"path: {DADOS}" if linha.startswith("path:") else linha
        for linha in texto.splitlines()
    ) + "\n"
    arq.write_text(novo, encoding="utf-8")

print((DADOS / "data_dobra1.yaml").read_text(encoding="utf-8"))

In [ ]:
# Cada linha das listas e um caminho relativo ao proprio arquivo de lista, e nao
# ao campo 'path': a Ultralytics troca o prefixo "./" pelo diretorio onde a
# lista esta. Como as listas ficam em dobras/, a forma correta sobe um nivel.
# Pacotes gerados antes dessa correcao trazem "./images/...", que resolve para
# dobras/images/ e nao existe. A normalizacao e idempotente: sobre um pacote
# correto ela nao altera nada.
ajustadas = 0
for arq in sorted((DADOS / "dobras").glob("*.txt")):
    atuais = arq.read_text(encoding="utf-8").split()
    novas = [f"./../images/{linha.rsplit('/', 1)[-1]}" for linha in atuais]
    if novas != atuais:
        arq.write_text("".join(f"{l}\n" for l in novas), encoding="utf-8")
        ajustadas += 1
print(f"listas normalizadas ....... {ajustadas} de 10")

# A verificacao e indispensavel: diante de um caminho inexistente a Ultralytics
# apenas avisa e segue com o que encontrou, de modo que um erro aqui produziria
# um treino sobre conjunto vazio em vez de uma falha evidente.
for arq in sorted((DADOS / "dobras").glob("*.txt")):
    for linha in arq.read_text(encoding="utf-8").split():
        destino = arq.parent / linha[2:]
        assert destino.exists(), f"{arq.name}: {linha} nao resolve"
print("caminhos das listas ....... todos resolvem")

In [ ]:
# Verificacao de integridade: os numeros tem de bater com o capitulo 2. Se nao
# baterem, o pacote enviado nao e o que foi auditado.
imagens = sorted((DADOS / "images").glob("*.jpg"))
rotulos = sorted((DADOS / "labels").glob("*.txt"))
classes = [l for l in (DADOS / "classes.txt").read_text(encoding="utf-8").split() if l]
caixas = sum(
    len([l for l in r.read_text(encoding="utf-8").splitlines() if l.strip()])
    for r in rotulos
)

print(f"imagens ... {len(imagens):>5}   esperado 1800")
print(f"rotulos ... {len(rotulos):>5}   esperado 1800")
print(f"classes ... {len(classes):>5}   esperado   12")
print(f"caixas .... {caixas:>5}   esperado 2757")

assert len(imagens) == len(rotulos) == 1800
assert len(classes) == 12
assert caixas == 2757

for dobra in DOBRAS:
    t = len((DADOS / "dobras" / f"dobra{dobra}_train.txt").read_text().split())
    v = len((DADOS / "dobras" / f"dobra{dobra}_val.txt").read_text().split())
    assert t + v == 1800, (dobra, t, v)
    print(f"dobra {dobra}: {t} treino + {v} validacao")

## 4. Retomada

O treino roda no disco local, que é rápido, e cada época espelha os pesos e as
métricas no Drive, que é persistente. Antes de iniciar uma execução, o notebook
procura no espelho um ponto salvo e, se houver, o traz de volta.

Um arquivo `concluido.txt` marca as execuções que terminaram, para que uma
reexecução do notebook não as repita.

In [ ]:
def caminho_espelho(nome):
    return RUNS_DRIVE / nome


def ja_concluida(nome):
    return (caminho_espelho(nome) / "concluido.txt").exists()


def restaura(nome):
    """Traz do Drive o ponto salvo da execucao, se existir.

    Devolve o caminho de last.pt quando ha de onde retomar, e None quando a
    execucao comeca do zero.
    """
    origem = caminho_espelho(nome)
    peso = origem / "weights" / "last.pt"
    if not peso.exists():
        return None
    destino = RUNS / nome
    if destino.exists():
        shutil.rmtree(destino)
    shutil.copytree(origem, destino)
    return destino / "weights" / "last.pt"


def espelha(trainer):
    """Copia pesos e metricas para o Drive. Chamada ao fim de cada epoca."""
    destino = caminho_espelho(Path(trainer.save_dir).name)
    for relativo in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml"):
        origem = Path(trainer.save_dir) / relativo
        if origem.exists():
            alvo = destino / relativo
            alvo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(origem, alvo)


def marca_concluida(nome):
    destino = caminho_espelho(nome)
    destino.mkdir(parents=True, exist_ok=True)
    (destino / "concluido.txt").write_text(
        f"ultralytics {ultralytics.__version__}\n"
        f"gpu {GPU}\n"
        f"concluida em {time.strftime('%Y-%m-%d %H:%M:%S')}\n",
        encoding="utf-8",
    )

In [ ]:
from ultralytics import YOLO


def treina(modelo, dobra, semente):
    """Executa uma celula do experimento: um modelo, uma dobra, uma semente.

    Idempotente: devolve imediatamente se a execucao ja terminou, e retoma do
    ultimo ponto salvo se foi interrompida.
    """
    nome = f"{modelo}_dobra{dobra}_s{semente}"

    if ja_concluida(nome):
        print(f"[ja concluida] {nome}")
        return nome

    ponto = restaura(nome)
    t0 = time.time()

    if ponto:
        print(f"[retomando]    {nome}")
        rede = YOLO(ponto)
        rede.add_callback("on_fit_epoch_end", espelha)
        resultado = rede.train(resume=True)
    else:
        print(f"[iniciando]    {nome}")
        rede = YOLO(f"{modelo}.pt")          # pesos pre-treinados em COCO
        rede.add_callback("on_fit_epoch_end", espelha)
        resultado = rede.train(
            data=str(DADOS / f"data_dobra{dobra}.yaml"),
            project=str(RUNS), name=nome, exist_ok=True,
            seed=semente, **HP,
        )

    espelha(rede.trainer)
    marca_concluida(nome)
    print(f"[concluida]    {nome} em {(time.time()-t0)/60:.0f} min")
    return nome

## 5. Fase 1 — os três modelos nas cinco dobras

Quinze execuções com semente fixa. Responde qual dos três modelos usar, e nada
mais: a escolha é feita aqui e o desempenho relatado para o modelo escolhido fica
ligeiramente otimista por isso — limitação registrada na Seção 2.9 e mitigada
pelas sementes da fase 2.

Interromper esta célula é seguro. Execute-a de novo e ela continua.

In [ ]:
for modelo in MODELOS:
    for dobra in DOBRAS:
        treina(modelo, dobra, SEMENTE_FASE1)

In [ ]:
import pandas as pd


def metricas(nome):
    """Le a ultima linha do results.csv de uma execucao."""
    arq = caminho_espelho(nome) / "results.csv"
    if not arq.exists():
        return None
    tabela = pd.read_csv(arq)
    tabela.columns = [c.strip() for c in tabela.columns]
    ultima = tabela.iloc[-1]
    return {
        "epocas": int(ultima["epoch"]),
        "mAP50": float(ultima["metrics/mAP50(B)"]),
        "mAP50-95": float(ultima["metrics/mAP50-95(B)"]),
        "precisao": float(ultima["metrics/precision(B)"]),
        "revocacao": float(ultima["metrics/recall(B)"]),
    }


linhas = []
for modelo in MODELOS:
    for dobra in DOBRAS:
        m = metricas(f"{modelo}_dobra{dobra}_s{SEMENTE_FASE1}")
        if m:
            linhas.append({"modelo": modelo, "dobra": dobra, "semente": SEMENTE_FASE1, **m})

fase1 = pd.DataFrame(linhas)
fase1.to_csv(RUNS_DRIVE / "fase1_execucoes.csv", index=False)

# Media e desvio padrao das cinco dobras. Nunca o melhor valor obtido: o melhor
# de cinco execucoes e uma estimativa enviesada.
resumo = fase1.groupby("modelo")[["mAP50", "mAP50-95", "precisao", "revocacao"]].agg(["mean", "std"])
print(resumo.round(4))
resumo.to_csv(RUNS_DRIVE / "fase1_resumo.csv")

## 6. Fase 2 — repetição com sementes distintas

Dez execuções do modelo escolhido. Separa o efeito do modelo do efeito da
inicialização aleatória; sem isso, não há como afirmar que a diferença observada
na fase 1 não foi sorte.

Preencha `MODELO_ESCOLHIDO` com o resultado da fase 1 antes de executar.

In [ ]:
MODELO_ESCOLHIDO = ""   # ex. "yolov8s" — decidido pela tabela da fase 1

assert MODELO_ESCOLHIDO in MODELOS, "preencha MODELO_ESCOLHIDO com o resultado da fase 1"

for semente in SEMENTES_FASE2:
    for dobra in DOBRAS:
        treina(MODELO_ESCOLHIDO, dobra, semente)

In [ ]:
linhas = []
for semente in [SEMENTE_FASE1] + SEMENTES_FASE2:
    for dobra in DOBRAS:
        m = metricas(f"{MODELO_ESCOLHIDO}_dobra{dobra}_s{semente}")
        if m:
            linhas.append({"modelo": MODELO_ESCOLHIDO, "dobra": dobra, "semente": semente, **m})

fase2 = pd.DataFrame(linhas)
fase2.to_csv(RUNS_DRIVE / "fase2_execucoes.csv", index=False)

print("por semente (media das 5 dobras)")
print(fase2.groupby("semente")[["mAP50", "mAP50-95"]].agg(["mean", "std"]).round(4))
print("\nagregado das 15 execucoes")
print(fase2[["mAP50", "mAP50-95", "precisao", "revocacao"]].agg(["mean", "std"]).round(4))

## 7. Próximas etapas

Fora do escopo deste notebook, e cada uma com seu próprio procedimento:

- **Desempenho por classe** cruzado com o número de exemplares, e matriz de
  confusão entre as doze classes (Seção 2.6.2). Lê os `results.csv` e as
  predições das execuções já gravadas no Drive.
- **Praga × inimigo natural** — taxa de confusão entre os dois grupos, calculada
  separadamente (Seção 2.6.2).
- **Tempo de inferência** — uma imagem por vez, vinte descartadas para
  aquecimento e cem cronometradas, decomposto em pré-processamento, rede e
  pós-processamento (Seção 2.6.3). Medido aqui na T4 e, com o modelo exportado
  para ONNX, na CPU da máquina local.